In [1]:
from syto.data.atlases.celfieish_atlases import CpGBetaCountsMethylationAtlas
import pandas as pd
import os
import json
import numpy as np

In [2]:
data_path = "~/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041"

In [2]:
data_path = "~/Data/Loyfer/SimulatedReads_205files_hg38/all_reads_transformed_uxm_prepared.parquet"

In [ ]:
# data = [pd.read_parquet(os.path.join(data_path,f"{x}.parquet")) for x in ["train", "valid", "test"] ]
# data = pd.concat(data)

In [ ]:
# data = pd.concat(data)

In [3]:
data = pd.read_parquet(data_path)

In [4]:
labels_dict_path = "../../../App/labels_dict.json"
with open(labels_dict_path, "r", encoding="utf-8") as f:
            # JSON keys are strings; convert to {int: str}
            labels_dict = json.load(f)

In [5]:
data.reset_index(inplace=True)

In [6]:
data.rename(columns = {"chr":"chromosome", "trimmed_start":"read_start", "trimmed_end":"read_end"}, inplace=True)

In [24]:
atlas = CpGBetaCountsMethylationAtlas.from_reads(data, reference_genome="hg38", atlas_name="UXMU25_hg38_l4", labels_dict=labels_dict, output_path="/home/luna.kuleuven.be/u0169940/Repos/syto/baselines/deconvolution/celfieish/UXMU25_hg38_l4.txt")

Aggregating groups: 100%|██████████| 32353/32353 [00:08<00:00, 3840.75group/s]


In [7]:
atlas = CpGBetaCountsMethylationAtlas(reference_genome="hg38", atlas_name="UXMU25_hg38_l1", atlas_path="/home/luna.kuleuven.be/u0169940/Repos/syto/baselines/deconvolution/celfieish/UXMU25_hg38_l1.txt")

In [8]:
from baselines.deconvolution.celfieish import build_celfieish_input, celfieish_deconvolution
from baselines.deconvolution.celfie import build_celfie_input, celfie_deconvolution

In [9]:
selected_reads = data[data["original_label"]==11].copy()

In [10]:
selected_reads.shape

(164030, 24)

In [11]:
# selected_reads_prepared = atlas.prepare_reads(selected_reads, atlas=atlas)
selected_reads_prepared = selected_reads

In [12]:
input_ceflieish = build_celfieish_input(selected_reads_prepared, atlas)

In [13]:
input_ceflie = build_celfie_input(selected_reads_prepared, atlas)

In [13]:
results = celfieish_deconvolution(input_ceflieish["matrices"], atlas.get_beta_for_regions(input_ceflieish["region_names"]), num_iterations=400, convergence_criteria=0.001)

In [14]:
np.set_printoptions(precision=3)
with np.printoptions(precision=3, suppress=True):
    print(results)

[0.016 0.004 0.083 0.008 0.018 0.007 0.048 0.015 0.001 0.006 0.013 0.592
 0.019 0.044 0.002 0.003 0.007 0.003 0.004 0.012 0.002 0.006 0.003 0.001
 0.003 0.005 0.005 0.02  0.004 0.001 0.006 0.    0.005 0.003 0.009 0.01
 0.003 0.005 0.004]


In [14]:
results = celfie_deconvolution(input_ceflie['x_meth'], input_ceflie['x_cov'], *atlas.get_meth_cov_for_regions(input_ceflie["region_names"]), num_iterations=400, convergence_criteria=0.001)

In [15]:
np.set_printoptions(precision=3)
with np.printoptions(precision=3, suppress=True):
    print(results)

[0.008 0.002 0.033 0.002 0.003 0.003 0.011 0.005 0.    0.001 0.004 0.865
 0.004 0.021 0.    0.    0.001 0.    0.    0.004 0.001 0.003 0.002 0.002
 0.    0.002 0.001 0.004 0.002 0.001 0.002 0.    0.001 0.001 0.002 0.005
 0.001 0.003 0.001]


In [31]:
np.set_printoptions(precision=3)
with np.printoptions(precision=3, suppress=True):
    print(results)

[0.    0.002 0.027 0.001 0.002 0.001 0.005 0.001 0.    0.001 0.001 0.917
 0.002 0.024 0.    0.    0.001 0.    0.001 0.001 0.    0.002 0.    0.
 0.001 0.001 0.001 0.002 0.001 0.    0.001 0.    0.001 0.001 0.    0.
 0.001 0.001 0.   ]
